<a href="https://colab.research.google.com/github/vzyhug/200123035_14DHTH07_DL/blob/main/CNN/CNN_multiclass_classification_prj1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## IMPORT LIB

In [3]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np

# Lấy danh sách các thiết bị GPU có sẵn
gpu_devices = tf.config.list_physical_devices('GPU')

if gpu_devices:
    print("✅ Đang chạy trên GPU!")
    for device in gpu_devices:
        print(f" - Chi tiết thiết bị: {device.name}")
else:
    print("❌ Không tìm thấy GPU. Đang chạy trên CPU (sẽ rất chậm).")
    print("💡 Mẹo: Vào menu 'Thời gian chạy' (Runtime) -> 'Thay đổi loại thời gian chạy' (Change runtime type) -> Chọn T4 GPU.")

✅ Đang chạy trên GPU!
 - Chi tiết thiết bị: /physical_device:GPU:0


## DATA

In [7]:
# Cài đặt thư viện kaggle
!pip install -q kaggle

# Bạn cần upload file kaggle.json (API token) của bạn lên Colab trước khi chạy bước này
# Tạo thư mục ẩn để chứa API key
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Tải dataset trực tiếp
!kaggle datasets download -d siddharthkumarsah/logo-dataset-2341-classes-and-167140-images

# Giải nén dữ liệu
!unzip -q logo-dataset-2341-classes-and-167140-images.zip -d logo_dataset

Dataset URL: https://www.kaggle.com/datasets/siddharthkumarsah/logo-dataset-2341-classes-and-167140-images
License(s): copyright-authors
Resuming from 1841299456 bytes (213489345 bytes left)...
100% 1.91G/1.91G [00:11<00:00, 18.2MB/s]



## PREPROCESSING

In [8]:
# Thay đổi đường dẫn này trỏ tới thư mục chứa các thư mục con của từng class (sau khi giải nén)
base_dir = 'logo_dataset/datasetcopy/trainandtest/train'

# 1. Khởi tạo Datagen cho tập Training (CÓ tăng cường dữ liệu)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,       # Xoay ảnh ngẫu nhiên tối đa 15 độ
    width_shift_range=0.1,   # Dịch chuyển ảnh theo chiều ngang 10%
    height_shift_range=0.1,  # Dịch chuyển ảnh theo chiều dọc 10%
    zoom_range=0.1,          # Phóng to/thu nhỏ ngẫu nhiên 10%
    fill_mode='nearest',     # Lấp đầy các pixel trống sinh ra do phép xoay/dịch
    validation_split=0.2     # Chia 20% cho tập validation
    # Lưu ý: Mình không dùng 'horizontal_flip=True' vì nhiều logo có chữ,
    # lật ngược ảnh sẽ làm chữ bị ngược, gây nhiễu cho mô hình.
)

# 2. Khởi tạo Datagen cho tập Validation (KHÔNG tăng cường, chỉ scale)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Tham số ảnh
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 64

print("Đang chuẩn bị tập Training:")
train_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

print("Đang chuẩn bị tập Validation:")
# Chú ý: Sử dụng val_datagen ở đây
val_generator = val_datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Đang chuẩn bị tập Training:
Found 133497 images belonging to 10 classes.
Đang chuẩn bị tập Validation:
Found 33369 images belonging to 10 classes.


## TRAIN

In [9]:
num_classes = 10

model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5), # Tránh overfitting
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# Bắt đầu huấn luyện (Có thể mất nhiều thời gian do dataset lớn)
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    12,845,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,943,946 (49.38 MB)

 Trainable params: 12,943,946 (49.38 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
 159/2086 ━━━━━━━━━━━━━━━━━━━━ 11:41 364ms/step - accuracy: 0.2904 - loss: 2.3599

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
import shutil

# 1. Kết nối với Google Drive của bạn (sẽ có popup yêu cầu quyền truy cập)
drive.mount('/content/drive')

# 2. Lưu mô hình trên Colab trước
model.save('logo_classification_model.keras')

# 3. Copy file mô hình vừa lưu sang Google Drive
# Lưu ý: '/content/drive/MyDrive/' là thư mục gốc Drive của bạn
shutil.copy('logo_classification_model.keras', '/content/drive/MyDrive/logo_classification_model.keras')

print("Đã lưu mô hình an toàn vào Google Drive của bạn!")

## SIMPLE TEST

In [ ]:
# Lấy một batch dữ liệu từ tập validation
val_images, val_labels = next(val_generator)

# Lấy ảnh đầu tiên trong batch
test_image = val_images[0]
true_label_index = np.argmax(val_labels[0])

# Mở rộng chiều để dự đoán (model cần input dạng batch: (1, 128, 128, 3))
img_array = np.expand_dims(test_image, axis=0)
predictions = model.predict(img_array)
predicted_class_index = np.argmax(predictions[0])

# Hiển thị ảnh
plt.imshow(test_image)
plt.title(f"Dự đoán: {predicted_class_index} | Thực tế: {true_label_index}")
plt.axis('off')
plt.show()